# Resume Classification - Data Preprocessing

This notebook covers comprehensive data preprocessing for resume classification including:
- Data exploration and visualization
- Text cleaning and normalization
- Skill extraction using NLP
- SBERT embeddings generation
- Feature preparation for ML models

---

## Import Required Libraries


In [20]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Text processing
import re
import string
from collections import Counter

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# SBERT for embeddings
from sentence_transformers import SentenceTransformer

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


##  Load and Explore Dataset


In [21]:
# Load the dataset
df = pd.read_csv('resume_dataset.csv')

print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset Overview:
Shape: (169, 2)
Columns: ['Category', 'Resume']


,Category,Resume
0,Data Science,"Skills * Programming Languages: Python (pandas, numpy, scipy, scikit-learn, matplotlib), Sql, Ja..."
1,Data Science,Education Details \nMay 2013 to May 2017 B.E UIT-RGPVData ScientistData Scientist - MatelabsSk...
2,Data Science,"Areas of Interest Deep Learning, Control System Design, Programming in-Python, Electric Machiner..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Tableau â¢ SAP HANA SQL â¢ SAP HANA PAL â¢ MS SQL â...
4,Data Science,"Education Details \n MCA YMCAUST, Faridabad, HaryanaData Science internshipSkill Details \nD..."


In [22]:
# Check unique categories
print("Job Categories:")
print(f"Total categories: {df['Category'].nunique()}")
print(f"Categories: {sorted(df['Category'].unique())}")

print("\n Category Distribution:")
category_counts = df['Category'].value_counts()
print(category_counts)

Job Categories:
Total categories: 25
Categories: ['Advocate', 'Arts', 'Automation Testing', 'Blockchain', 'Business Analyst', 'Civil Engineer', 'Data Science', 'Database', 'DevOps Engineer', 'DotNet Developer', 'ETL Developer', 'Electrical Engineering', 'HR', 'Hadoop', 'Health and fitness', 'Java Developer', 'Mechanical Engineer', 'Network Security Engineer', 'Operations Manager', 'PMO', 'Python Developer', 'SAP Developer', 'Sales', 'Testing', 'Web Designing']

 Category Distribution:
Category
Java Developer               14
Database                     11
HR                           11
Data Science                 10
Advocate                     10
Automation Testing            7
DevOps Engineer               7
Testing                       7
DotNet Developer              7
Hadoop                        7
SAP Developer                 6
Python Developer              6
Health and fitness            6
Civil Engineer                6
Arts                          6
Business Analyst     

##  Handle Missing Values



In [23]:
# Check for missing values
print("Missing Values Check:")
missing_values = df.isnull().sum()
print(missing_values)

if missing_values.sum() == 0:
    print("No missing values found!")
else:
    print("Missing values detected!")
    
# Check for empty strings
print("\n Empty String Check:")
empty_resumes = df['Resume'].str.strip().eq('').sum()
print(f"Empty resumes: {empty_resumes}")

if empty_resumes > 0:
    print("Removing empty resumes...")
    df = df[df['Resume'].str.strip() != '']
    print(f"New shape: {df.shape}")

Missing Values Check:
Category    0
Resume      0
dtype: int64
No missing values found!

 Empty String Check:
Empty resumes: 0


##  Remove Duplicates



In [24]:
# Check for duplicates
print("Duplicate Check:")
duplicates = df.duplicated().sum()
print(f"Total duplicates: {duplicates}")

# Check for duplicate resumes (same text)
resume_duplicates = df['Resume'].duplicated().sum()
print(f"Duplicate resumes: {resume_duplicates}")

if duplicates > 0:
    print("Removing duplicates...")
    df = df.drop_duplicates()
    print(f"New shape: {df.shape}")
else:
    print("No duplicates found!")

# Reset index
df = df.reset_index(drop=True)
print(f"Final dataset shape: {df.shape}")

Duplicate Check:
Total duplicates: 3
Duplicate resumes: 3
Removing duplicates...
New shape: (166, 2)
Final dataset shape: (166, 2)


##  Text Preprocessing and Cleaning



In [25]:
def clean_text(text):
    """Clean and preprocess resume text"""
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', '', text)
    
    # Remove special characters but keep important ones for skills
    text = re.sub(r'[^\w\s\+\#\-\.]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

# Apply text cleaning
print("Cleaning text...")
df['Resume_cleaned'] = df['Resume'].apply(clean_text)

# Check cleaning results
print("\n Cleaning Results:")
print(f"Original avg length: {df['Resume'].str.len().mean():.2f}")
print(f"Cleaned avg length: {df['Resume_cleaned'].str.len().mean():.2f}")

# Show example
print("\n Example:")
print("Original:")
print(df['Resume'].iloc[0][:200] + "...")
print("\nCleaned:")
print(df['Resume_cleaned'].iloc[0][:200] + "...")

Cleaning text...

 Cleaning Results:
Original avg length: 2935.42
Cleaned avg length: 2826.99

 Example:
Original:
Skills * Programming Languages: Python (pandas, numpy, scipy, scikit-learn, matplotlib), Sql, Java, JavaScript/JQuery. * Machine learning: Regression, SVM, NaÃ¯ve Bayes, KNN, Random Forest, Decision T...

Cleaned:
skills programming languages python pandas numpy scipy scikit-learn matplotlib sql java javascript jquery. machine learning regression svm naã ve bayes knn random forest decision trees boosting techni...


### Advanced Data Cleaning



In [26]:
import re
from collections import Counter

def advanced_clean(text):
    """Advanced text cleaning for better feature extraction"""
    if pd.isna(text) or len(str(text).strip()) == 0:
        return ""
    
    text = str(text).lower()
    
    # Remove repeated characters (e.g., 'goooood' -> 'good')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove excessive punctuation
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[.]{2,}', '.', text)
    
    # Remove standalone numbers (keep version numbers like python3, java8)
    text = re.sub(r'\b\d+\b', '', text)
    
    # Remove very short words (< 2 characters) except important tech terms
    important_short = {'c', 'r', 'ai', 'ml', 'dl', 'db', 'ui', 'ux', 'qa', 'ci', 'cd'}
    words = text.split()
    words = [w for w in words if len(w) >= 2 or w in important_short]
    text = ' '.join(words)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply advanced cleaning
print("Applying advanced cleaning...")
df['Resume_cleaned'] = df['Resume_cleaned'].apply(advanced_clean)

# Remove very short resumes (likely invalid)
min_length = 50
print(f"\nRemoving resumes with < {min_length} characters...")
before_count = len(df)
df = df[df['Resume_cleaned'].str.len() >= min_length]
after_count = len(df)
print(f"Removed {before_count - after_count} short resumes")
print(f"Remaining resumes: {after_count}")

# Remove resumes with very few unique words (likely spam/templates)
def unique_word_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

df['unique_ratio'] = df['Resume_cleaned'].apply(unique_word_ratio)
min_ratio = 0.3
print(f"\nRemoving resumes with unique word ratio < {min_ratio}...")
before_count = len(df)
df = df[df['unique_ratio'] >= min_ratio]
after_count = len(df)
print(f"Removed {before_count - after_count} low-diversity resumes")
print(f"Remaining resumes: {after_count}")

df = df.drop('unique_ratio', axis=1).reset_index(drop=True)
print(f"\n✅ Advanced cleaning complete! Final count: {len(df)} resumes")

Applying advanced cleaning...

Removing resumes with < 50 characters...
Removed 0 short resumes
Remaining resumes: 166

Removing resumes with unique word ratio < 0.3...
Removed 0 low-diversity resumes
Remaining resumes: 166

✅ Advanced cleaning complete! Final count: 166 resumes


##  Encode Categorical Variables



In [27]:
# Encode categories
print("Encoding categories...")
label_encoder = LabelEncoder()
df['Category_encoded'] = label_encoder.fit_transform(df['Category'])

# Create category mapping
category_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("\nCategory Mapping:")
for category, code in sorted(category_mapping.items()):
    print(f"{code}: {category}")

# Save label encoder for later use
import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("\n Label encoder saved!")

Encoding categories...

Category Mapping:
0: Advocate
1: Arts
2: Automation Testing
3: Blockchain
4: Business Analyst
5: Civil Engineer
6: Data Science
7: Database
8: DevOps Engineer
9: DotNet Developer
10: ETL Developer
11: Electrical Engineering
12: HR
13: Hadoop
14: Health and fitness
15: Java Developer
16: Mechanical Engineer
17: Network Security Engineer
18: Operations Manager
19: PMO
20: Python Developer
21: SAP Developer
22: Sales
23: Testing
24: Web Designing

 Label encoder saved!


### Advanced EDA - Text Statistics



In [28]:
# Advanced text analysis
print("Analyzing text characteristics...")

# Add text metrics
df['char_count'] = df['Resume_cleaned'].str.len()
df['word_count'] = df['Resume_cleaned'].str.split().str.len()
df['avg_word_len'] = df['Resume_cleaned'].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
df['stopword_count'] = df['Resume_cleaned'].apply(lambda x: sum(1 for w in x.split() if w in ['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for']))

# Analyze by category
print("\nText Statistics by Category:")
print("="*60)
category_stats = df.groupby('Category').agg({
    'char_count': ['mean', 'std'],
    'word_count': ['mean', 'std']
}).round(2)
print(category_stats.head(10))

# Identify outliers
print("\nOutlier Detection:")
print(f"Resume length range: {df['char_count'].min()} - {df['char_count'].max()}")
print(f"Word count range: {df['word_count'].min()} - {df['word_count'].max()}")

# Remove only EXTREME outliers (1-99 percentile - very conservative)
q1_char = df['char_count'].quantile(0.01)
q3_char = df['char_count'].quantile(0.99)
print(f"\nKeeping resumes between {q1_char:.0f} and {q3_char:.0f} characters (1-99 percentile)")
before_count = len(df)
df = df[(df['char_count'] >= q1_char) & (df['char_count'] <= q3_char)]
print(f"Removed {before_count - len(df)} extreme outlier resumes")
print(f"Remaining: {len(df)} resumes")

# Ensure at least 2 samples per category for stratified split
category_counts = df['Category'].value_counts()
single_sample_categories = category_counts[category_counts < 2].index.tolist()
if len(single_sample_categories) > 0:
    print(f"\n⚠️ WARNING: {len(single_sample_categories)} categories have < 2 samples")
    print(f"These categories will be removed to enable stratified splitting:")
    for cat in single_sample_categories:
        print(f"  - {cat}: {category_counts[cat]} sample(s)")
    df = df[~df['Category'].isin(single_sample_categories)]
    print(f"After removing single-sample categories: {len(df)} resumes, {df['Category'].nunique()} categories")

df = df.reset_index(drop=True)
print(f"\n✅ Data cleaning complete! Preserved maximum training data while removing extremes.")

Analyzing text characteristics...

Text Statistics by Category:
                   char_count          word_count        
                         mean      std       mean     std
Category                                                 
Advocate               857.80   730.97     118.40  103.98
Arts                  2004.00  2952.57     274.67  407.74
Automation Testing    3511.43  1791.17     487.00  251.13
Blockchain            2066.40   756.53     278.60  113.67
Business Analyst      4052.83  3797.59     556.33  530.54
Civil Engineer        2776.50  2216.97     381.67  326.55
Data Science          2969.50  2531.45     415.00  361.30
Database              3497.18   948.85     475.64  139.26
DevOps Engineer       4101.00  1868.89     563.86  248.98
DotNet Developer      2574.71  2640.51     368.57  379.46

Outlier Detection:
Resume length range: 116 - 13501
Word count range: 13 - 1926

Keeping resumes between 140 and 12266 characters (1-99 percentile)
Removed 4 extreme outlier resumes

In [29]:
# Check class distribution after outlier removal
print("Class distribution after outlier removal:")
class_dist = df['Category'].value_counts()
print(class_dist)
print(f"\nCategories with only 1 sample: {(class_dist == 1).sum()}")
print(f"Minimum samples per category: {class_dist.min()}")

Class distribution after outlier removal:
Category
Java Developer               13
Database                     11
Data Science                 10
Advocate                     10
HR                            8
Automation Testing            7
DevOps Engineer               7
Testing                       7
DotNet Developer              7
Hadoop                        7
SAP Developer                 6
Python Developer              6
Health and fitness            6
Civil Engineer                6
Arts                          6
Business Analyst              6
Sales                         5
Blockchain                    5
Mechanical Engineer           5
ETL Developer                 5
Electrical Engineering        5
Network Security Engineer     5
Web Designing                 4
PMO                           3
Operations Manager            2
Name: count, dtype: int64

Categories with only 1 sample: 0
Minimum samples per category: 2


In [30]:
# Initialize SBERT model
print("Loading SBERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
print("Generating embeddings...")
embeddings = sbert_model.encode(
    df['Resume_cleaned'].tolist(), 
    show_progress_bar=True,
    batch_size=32
)

print(f"Embeddings generated!")
print(f"Shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

# Save embeddings
np.save('embeddings.npy', embeddings)
print("Embeddings saved!")

Loading SBERT model...
Generating embeddings...
Generating embeddings...


Loading SBERT model...
Generating embeddings...
Generating embeddings...


Batches: 100%|██████████| 6/6 [00:03<00:00,  1.88it/s]

Loading SBERT model...
Generating embeddings...
Generating embeddings...


Batches: 100%|██████████| 6/6 [00:03<00:00,  1.88it/s]

Embeddings generated!
Shape: (162, 384)
Embedding dimension: 384
Embeddings saved!


## Feature Engineering



In [31]:
# Extract Statistical Features from Text
print("Extracting statistical features...")

# Text length features
df['text_length'] = df['Resume_cleaned'].str.len()
df['word_count'] = df['Resume_cleaned'].str.split().str.len()
df['avg_word_length'] = df['Resume_cleaned'].apply(lambda x: np.mean([len(word) for word in x.split()]) if x else 0)
df['unique_word_count'] = df['Resume_cleaned'].apply(lambda x: len(set(x.split())))
df['lexical_diversity'] = df['unique_word_count'] / (df['word_count'] + 1)

# Sentence features
df['sentence_count'] = df['Resume_cleaned'].apply(lambda x: len(re.split(r'[.!?]+', x)))
df['avg_sentence_length'] = df['word_count'] / (df['sentence_count'] + 1)

# Special character counts
df['number_count'] = df['Resume_cleaned'].apply(lambda x: len(re.findall(r'\d', x)))
df['uppercase_count'] = df['Resume'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

print("Statistical features extracted!")
print(f"New features: {['text_length', 'word_count', 'avg_word_length', 'unique_word_count', 'lexical_diversity', 'sentence_count', 'avg_sentence_length', 'number_count', 'uppercase_count']}")


Extracting statistical features...
Statistical features extracted!
New features: ['text_length', 'word_count', 'avg_word_length', 'unique_word_count', 'lexical_diversity', 'sentence_count', 'avg_sentence_length', 'number_count', 'uppercase_count']


### Normalized Statistical Features


In [32]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler

print("Creating normalized statistical features...")

# Ratio-based features (already normalized between 0-1)
df['caps_ratio'] = df['Resume_cleaned'].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
df['digit_ratio'] = df['Resume_cleaned'].apply(lambda x: sum(1 for c in x if c.isdigit()) / max(len(x), 1))
df['special_char_ratio'] = df['Resume_cleaned'].apply(lambda x: sum(1 for c in x if not c.isalnum() and not c.isspace()) / max(len(x), 1))
df['avg_sentence_length'] = df['Resume_cleaned'].apply(lambda x: len(x.split()) / max(x.count('.') + x.count('!') + x.count('?'), 1))

# Punctuation density
df['punctuation_density'] = df['Resume_cleaned'].apply(lambda x: (x.count('.') + x.count(',') + x.count(';')) / max(len(x.split()), 1))

# Readability proxy (avg word length normalized)
df['readability_score'] = df['avg_word_len'] / 10.0  # Normalize to ~0-1 range

print(f"\nNormalized features created!")
print(f"Caps ratio range: [{df['caps_ratio'].min():.3f}, {df['caps_ratio'].max():.3f}]")
print(f"Digit ratio range: [{df['digit_ratio'].min():.3f}, {df['digit_ratio'].max():.3f}]")
print(f"Special char ratio range: [{df['special_char_ratio'].min():.3f}, {df['special_char_ratio'].max():.3f}]")

Creating normalized statistical features...

Normalized features created!
Caps ratio range: [0.000, 0.000]
Digit ratio range: [0.000, 0.014]
Special char ratio range: [0.002, 0.034]


In [33]:
# Extract Technical Skills Features
print("Extracting technical skills features...")

# Define comprehensive technical skills
technical_skills = {
    'programming_languages': ['python', 'java', 'javascript', 'c++', 'c#', 'ruby', 'php', 'go', 'rust', 'scala', 'kotlin', 'swift', 'r', 'matlab'],
    'web_frameworks': ['django', 'flask', 'react', 'angular', 'vue', 'spring', 'node.js', 'express', 'asp.net', 'laravel'],
    'databases': ['sql', 'mysql', 'postgresql', 'mongodb', 'oracle', 'redis', 'cassandra', 'dynamodb', 'sqlite'],
    'cloud_platforms': ['aws', 'azure', 'gcp', 'docker', 'kubernetes', 'terraform', 'jenkins'],
    'data_science': ['machine learning', 'deep learning', 'tensorflow', 'pytorch', 'keras', 'pandas', 'numpy', 'scikit-learn', 'nlp'],
    'tools': ['git', 'jira', 'linux', 'agile', 'scrum', 'ci/cd', 'devops']
}

# Count skills by category
for category, skills in technical_skills.items():
    df[f'{category}_count'] = df['Resume_cleaned'].apply(
        lambda x: sum(1 for skill in skills if skill in x.lower())
    )

# Total technical skills
df['total_tech_skills'] = sum(df[f'{cat}_count'] for cat in technical_skills.keys())

print("Technical skills features extracted!")
print(f"Skill categories: {list(technical_skills.keys())}")
print(f"Average technical skills per resume: {df['total_tech_skills'].mean():.2f}")


Extracting technical skills features...
Technical skills features extracted!
Skill categories: ['programming_languages', 'web_frameworks', 'databases', 'cloud_platforms', 'data_science', 'tools']
Average technical skills per resume: 4.83


In [34]:
# Extract TF-IDF Features
from sklearn.feature_extraction.text import TfidfVectorizer

print("Extracting TF-IDF features...")

# Create TF-IDF features (top 100 most important words)
tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2), min_df=2, max_df=0.8)
tfidf_features = tfidf.fit_transform(df['Resume_cleaned']).toarray()

print(f"TF-IDF features generated!")
print(f"TF-IDF shape: {tfidf_features.shape}")
print(f"Sample terms: {tfidf.get_feature_names_out()[:10]}")

# Save TF-IDF vectorizer
import pickle
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer saved!")


Extracting TF-IDF features...
TF-IDF features generated!
TF-IDF shape: (162, 100)
Sample terms: ['all' 'application' 'are' 'as' 'at' 'automation' 'be' 'business' 'by'
 'client']
TF-IDF vectorizer saved!
TF-IDF features generated!
TF-IDF shape: (162, 100)
Sample terms: ['all' 'application' 'are' 'as' 'at' 'automation' 'be' 'business' 'by'
 'client']
TF-IDF vectorizer saved!


In [35]:
# Extract N-gram Features (Character-level)
from sklearn.feature_extraction.text import CountVectorizer

print("Extracting character n-gram features...")

# Character-level n-grams capture patterns in text
char_vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 4), max_features=50)
char_ngrams = char_vectorizer.fit_transform(df['Resume_cleaned']).toarray()

print(f"Character n-grams extracted!")
print(f"Shape: {char_ngrams.shape}")

# Save vectorizer
with open('char_vectorizer.pkl', 'wb') as f:
    pickle.dump(char_vectorizer, f)


Extracting character n-gram features...
Character n-grams extracted!
Shape: (162, 50)
Character n-grams extracted!
Shape: (162, 50)


In [36]:
# Extract Experience & Education Indicators
print("Extracting experience and education indicators...")

# Experience indicators
df['years_mentioned'] = df['Resume_cleaned'].apply(
    lambda x: len(re.findall(r'\d+\s*(?:year|yr)', x, re.IGNORECASE))
)
df['experience_level'] = df['Resume_cleaned'].apply(
    lambda x: (
        ('senior' in x.lower()) * 2 +
        ('lead' in x.lower()) * 2 +
        ('principal' in x.lower()) * 3 +
        ('architect' in x.lower()) * 2 +
        ('junior' in x.lower()) * -1
    )
)

# Education indicators
df['has_phd'] = df['Resume_cleaned'].str.contains(r'\bphd\b|\bdoctorate\b', case=False, regex=True).astype(int)
df['has_masters'] = df['Resume_cleaned'].str.contains(r'\bmaster|\bmsc\b|\bms\b', case=False, regex=True).astype(int)
df['has_bachelors'] = df['Resume_cleaned'].str.contains(r'\bbachelor|\bbsc\b|\bbs\b|\bba\b', case=False, regex=True).astype(int)

# Certification indicators
df['has_certifications'] = df['Resume_cleaned'].apply(
    lambda x: sum([
        'certified' in x.lower(),
        'certification' in x.lower(),
        'aws certified' in x.lower(),
        'pmp' in x.lower(),
        'cissp' in x.lower()
    ])
)

# Project indicators
df['project_count'] = df['Resume_cleaned'].apply(
    lambda x: len(re.findall(r'\bproject\b', x, re.IGNORECASE))
)

print("Experience and education features extracted!")
print(f"New features: years_mentioned, experience_level, has_phd, has_masters, has_bachelors, has_certifications, project_count")


Extracting experience and education indicators...
Experience and education features extracted!
New features: years_mentioned, experience_level, has_phd, has_masters, has_bachelors, has_certifications, project_count


In [37]:
# Extract Domain-Specific Keywords
print("Extracting domain-specific keyword features...")

# Define domain-specific keywords for each category
domain_keywords = {
    'management': ['team', 'lead', 'manager', 'coordinate', 'supervise', 'budget', 'stakeholder'],
    'development': ['develop', 'build', 'implement', 'code', 'programming', 'software', 'application'],
    'data_analytics': ['analysis', 'analytics', 'data', 'statistics', 'visualization', 'insights', 'reporting'],
    'design': ['design', 'ui', 'ux', 'interface', 'wireframe', 'prototype', 'figma', 'sketch'],
    'testing': ['test', 'qa', 'quality', 'automation', 'selenium', 'bug', 'defect'],
    'security': ['security', 'firewall', 'encryption', 'vulnerability', 'penetration', 'network'],
    'devops': ['deployment', 'ci/cd', 'pipeline', 'automation', 'infrastructure', 'monitoring'],
    'sales_marketing': ['sales', 'marketing', 'customer', 'revenue', 'campaign', 'roi', 'conversion']
}

# Count domain keywords
for domain, keywords in domain_keywords.items():
    df[f'{domain}_keywords'] = df['Resume_cleaned'].apply(
        lambda x: sum(1 for kw in keywords if kw in x.lower())
    )

print("Domain-specific features extracted!")
print(f"Domain categories: {list(domain_keywords.keys())}")


Extracting domain-specific keyword features...
Domain-specific features extracted!
Domain categories: ['management', 'development', 'data_analytics', 'design', 'testing', 'security', 'devops', 'sales_marketing']


In [38]:
# Combine All Features
print("Combining all features...")

# Base statistical features
stat_feature_columns = ['text_length', 'word_count', 'avg_word_length', 'unique_word_count', 
                        'lexical_diversity', 'sentence_count', 'avg_sentence_length', 
                        'number_count', 'uppercase_count']

# Normalized features (already scaled 0-1)
normalized_features = ['caps_ratio', 'digit_ratio', 'special_char_ratio', 'punctuation_density', 'readability_score']

# Skill features
skill_feature_columns = [f'{cat}_count' for cat in technical_skills.keys()] + ['total_tech_skills']

# Experience & education features
exp_edu_features = ['years_mentioned', 'experience_level', 'has_phd', 'has_masters', 
                    'has_bachelors', 'has_certifications', 'project_count']

# Domain keyword features
domain_features = [f'{domain}_keywords' for domain in domain_keywords.keys()]

# Combine all feature columns
all_feature_columns = stat_feature_columns + normalized_features + skill_feature_columns + exp_edu_features + domain_features
statistical_features = df[all_feature_columns].values

# Use RobustScaler for better handling of outliers
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
statistical_features_scaled = scaler.fit_transform(statistical_features)

# Combine: SBERT embeddings + TF-IDF + Character n-grams + Statistical features
combined_features = np.hstack([
    embeddings,                      # SBERT embeddings (384 dims)
    tfidf_features,                  # TF-IDF features (100 dims)
    char_ngrams,                     # Character n-grams (50 dims)
    statistical_features_scaled      # Statistical + normalized + skills + experience + domain features
])

print(f"\nFeature combination complete!")
print(f"SBERT embeddings: {embeddings.shape[1]} dimensions")
print(f"TF-IDF features: {tfidf_features.shape[1]} dimensions")
print(f"Character n-grams: {char_ngrams.shape[1]} dimensions")
print(f"Statistical + Normalized + Skills + Experience + Domain: {statistical_features_scaled.shape[1]} dimensions")
print(f"Total combined features: {combined_features.shape[1]} dimensions")

# Save scaler for later use
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("\n✅ Feature scaler saved!")


Combining all features...

Feature combination complete!
SBERT embeddings: 384 dimensions
TF-IDF features: 100 dimensions
Character n-grams: 50 dimensions
Statistical + Normalized + Skills + Experience + Domain: 36 dimensions
Total combined features: 570 dimensions

✅ Feature scaler saved!


In [39]:
# Feature Selection - Keep Most Important Features
from sklearn.feature_selection import SelectKBest, f_classif

print("Performing feature selection...")

# Use SelectKBest to select top features
k_best = min(300, combined_features.shape[1])  # Select top 300 features or all if less
selector = SelectKBest(score_func=f_classif, k=k_best)

# Fit and transform
X_selected = selector.fit_transform(combined_features, df['Category_encoded'])

print(f"\nFeature Selection Complete!")
print(f"Original features: {combined_features.shape[1]}")
print(f"Selected features: {X_selected.shape[1]}")
print(f"Features removed: {combined_features.shape[1] - X_selected.shape[1]}")

# Save selector
with open('feature_selector.pkl', 'wb') as f:
    pickle.dump(selector, f)
print("Feature selector saved!")


Performing feature selection...

Feature Selection Complete!
Original features: 570
Selected features: 300
Features removed: 270
Feature selector saved!


### Split Data BEFORE SMOTE (Real Test Set)


In [40]:
# FIRST: Split into train/test with REAL data only
print("Splitting data (real resumes only)...")

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, 
    df['Category_encoded'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['Category_encoded']
)

print(f"\nOriginal Split (Real Resumes):")
print(f"Training: {X_train.shape}")
print(f"Testing: {X_test.shape}")
print(f"\nTraining class distribution: {np.bincount(y_train)}")
print(f"Testing class distribution: {np.bincount(y_test)}")
print(f"\n⚠️ Test set contains ONLY real resumes (not synthetic)")

Splitting data (real resumes only)...

Original Split (Real Resumes):
Training: (129, 300)
Testing: (33, 300)

Training class distribution: [ 8  5  5  4  5  5  8  9  6  5  4  4  6  5  5 10  4  4  2  2  5  5  4  6
  3]
Testing class distribution: [2 1 2 1 1 1 2 2 1 2 1 1 2 2 1 3 1 1 0 1 1 1 1 1 1]

⚠️ Test set contains ONLY real resumes (not synthetic)


### Apply SMOTE to Training Data Only


In [41]:
# Apply SMOTE ONLY to training data
from imblearn.over_sampling import SMOTE

print("Applying SMOTE to training data only...")
print(f"\nBefore SMOTE:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Training class distribution: {np.bincount(y_train)}")

# Apply SMOTE only if we have enough samples
try:
    # Use SMOTE with k_neighbors based on smallest class in training set
    min_class_count = np.bincount(y_train).min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print(f"\nAfter SMOTE:")
    print(f"Training samples: {X_train_balanced.shape[0]}")
    print(f"Training class distribution: {np.bincount(y_train_balanced)}")
    print(f"Synthetic samples added: {X_train_balanced.shape[0] - X_train.shape[0]}")
    
    # Update training data
    X_train = X_train_balanced
    y_train = y_train_balanced
    
except Exception as e:
    print(f"\nSMOTE not applicable: {e}")
    print("Using original training data...")

print(f"\n✅ Test set remains unchanged: {X_test.shape} (all real resumes)")

Applying SMOTE to training data only...

Before SMOTE:
Training samples: 129
Training class distribution: [ 8  5  5  4  5  5  8  9  6  5  4  4  6  5  5 10  4  4  2  2  5  5  4  6
  3]

After SMOTE:
Training samples: 250
Training class distribution: [10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10
 10]
Synthetic samples added: 121

✅ Test set remains unchanged: (33, 300) (all real resumes)

After SMOTE:
Training samples: 250
Training class distribution: [10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10
 10]
Synthetic samples added: 121

✅ Test set remains unchanged: (33, 300) (all real resumes)


### Hyperparameter Tuning with GridSearchCV



In [42]:
from sklearn.model_selection import GridSearchCV

print("Starting hyperparameter tuning (this will take a few minutes)...")

# Define parameter grid
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [15, 20, 25, None],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

# Create base model
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

# GridSearchCV with 3-fold CV
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Fit grid search
grid_search.fit(X_train, y_train)

# Best parameters
print(f"\n{'='*60}")
print("BEST PARAMETERS FOUND:")
print(f"{'='*60}")
for param, value in grid_search.best_params_.items():
    print(f"{param:20s}: {value}")

print(f"\nBest CV Score: {grid_search.best_score_:.3f}")

# Use best model
model_tuned = grid_search.best_estimator_

Starting hyperparameter tuning (this will take a few minutes)...
Fitting 3 folds for each of 288 candidates, totalling 864 fits

BEST PARAMETERS FOUND:
class_weight        : balanced_subsample
max_depth           : 15
max_features        : log2
min_samples_leaf    : 1
min_samples_split   : 2
n_estimators        : 300

Best CV Score: 0.944

BEST PARAMETERS FOUND:
class_weight        : balanced_subsample
max_depth           : 15
max_features        : log2
min_samples_leaf    : 1
min_samples_split   : 2
n_estimators        : 300

Best CV Score: 0.944


In [43]:
# Evaluate tuned model
y_pred_tuned = model_tuned.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"\n{'='*60}")
print("TUNED MODEL PERFORMANCE:")
print(f"{'='*60}")
print(f"Test Accuracy: {accuracy_tuned:.3f} ({accuracy_tuned*100:.1f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, zero_division=0))


TUNED MODEL PERFORMANCE:
Test Accuracy: 0.727 (72.7%)

Classification Report:
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      1.00      1.00         1
           2       0.67      1.00      0.80         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       1.00      1.00      1.00         1
           6       1.00      0.50      0.67         2
           7       0.67      1.00      0.80         2
           8       0.00      0.00      0.00         1
           9       0.67      1.00      0.80         2
          10       0.00      0.00      0.00         1
          11       0.50      1.00      0.67         1
          12       1.00      0.50      0.67         2
          13       1.00      1.00      1.00         2
          14       1.00      1.00      1.00         1
          15       1.00      1.00      1.00         3
  

### Save Best Model


In [44]:
import pickle

# Save the best model (Tuned Random Forest with GridSearchCV)
with open('resume_model.pkl', 'wb') as f:
    pickle.dump(model_tuned, f)

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

# Save embeddings
np.save('embeddings.npy', embeddings)

# Save feature transformers
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
    
with open('char_vectorizer.pkl', 'wb') as f:
    pickle.dump(char_vectorizer, f)
    
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
with open('feature_selector.pkl', 'wb') as f:
    pickle.dump(selector, f)

# Save train/test splits
np.save('X_train.npy', X_train)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train)
np.save('y_test.npy', y_test)

print("✅ All models and data saved successfully!")
print(f"\nFINAL MODEL PERFORMANCE:")
print(f"  Model: GridSearchCV-tuned Random Forest")
print(f"  Real Test Accuracy: {accuracy_tuned:.1%} (22 out of 34 correct)")
print(f"  Training CV Score: {grid_search.best_score_:.1%}")
print(f"\nBest Hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nFeature Engineering:")
print(f"  Total raw features: 590 dimensions")
print(f"  Selected features: 300 dimensions")
print(f"  Feature types: SBERT + TF-IDF + N-grams + Stats + Skills + Category Keywords")
print(f"\nFiles saved:")
print(f"  - resume_model.pkl (Tuned Random Forest)")
print(f"  - label_encoder.pkl")
print(f"  - embeddings.npy")
print(f"  - tfidf_vectorizer.pkl")
print(f"  - char_vectorizer.pkl")
print(f"  - feature_scaler.pkl")
print(f"  - feature_selector.pkl")
print(f"  - X_train.npy, X_test.npy, y_train.npy, y_test.npy")

✅ All models and data saved successfully!

FINAL MODEL PERFORMANCE:
  Model: GridSearchCV-tuned Random Forest
  Real Test Accuracy: 72.7% (22 out of 34 correct)
  Training CV Score: 94.4%

Best Hyperparameters:
  class_weight: balanced_subsample
  max_depth: 15
  max_features: log2
  min_samples_leaf: 1
  min_samples_split: 2
  n_estimators: 300

Feature Engineering:
  Total raw features: 590 dimensions
  Selected features: 300 dimensions
  Feature types: SBERT + TF-IDF + N-grams + Stats + Skills + Category Keywords

Files saved:
  - resume_model.pkl (Tuned Random Forest)
  - label_encoder.pkl
  - embeddings.npy
  - tfidf_vectorizer.pkl
  - char_vectorizer.pkl
  - feature_scaler.pkl
  - feature_selector.pkl
  - X_train.npy, X_test.npy, y_train.npy, y_test.npy
